# Learned placement pilot (v5)

Train three small RL planner policies on local CPU simulations, then compare
frozen policies with heuristics/search and a supervised control. This is **one-step
contextual-bandit RL selecting a complete feasible plan**, not a controller that
replans during failures. There is no actual distributed model training, Ray, GPU
or Z3. See `docs/PILOT_V5.md` for the exact objective and experiment.

The simulation horizon increases from 8× to **32×** reference time. Deadlines and
budgets do not change. The real wall-clock limit remains 30 minutes per invocation.


In [ ]:
from pathlib import Path
import sys
import json
ROOT = Path.cwd()
if not (ROOT / "learnedplacement").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "learnedplacement").is_dir()
sys.path.insert(0, str(ROOT))

# Install once from a terminal: python -m pip install -e '.[notebook,learning]'
import numpy as np
import pandas as pd
from learnedplacement.runner import read_json, status
from learnedplacement.learning_runner import run, export, validate_config
from learnedplacement.expanded_runner import plot_results


## Editable settings

Fresh output `pilot_v5`; do not reuse v4 results. Default: 192 training cases,
48 validation and 48 test cases. All profiles of one hardware instance share its
split. Three RL seeds each train 2,048 episodes × 8 simulated failure rollouts.
Evaluation uses 256 fresh paired traces per case and method (270,336 evaluations).
A full sweep may require several sessions; rerun unchanged to resume.

Only TRAIN cases update weights. Final policies freeze before either validation
or test evaluation. No best seed/checkpoint is selected using those outcomes.


In [ ]:
RUN_NAME = "pilot_v5"
MAX_MINUTES = 30
CONFIG = read_json(ROOT / "configs" / "pilot_v5.json")
CONFIG["simulation_horizon_factor"] = 32.0
CONFIG["planning_traces"] = 32
CONFIG["evaluation_traces"] = 256
CONFIG["learning"]["episodes"] = 2048
CONFIG["learning"]["rollouts_per_episode"] = 8
CONFIG["learning"]["action_count"] = 64
CONFIG["learning"]["learning_rate"] = 0.001
CONFIG["learning"]["entropy"] = 0.01
CONFIG["learning"]["penalties"] = {"cost": 20.0, "deadline": 20.0, "censor": 100.0}

# Optional smaller first experiment; edit BEFORE starting a new run:
# CONFIG["active_nodes"] = [4, 8]
# CONFIG["expanded_profiles"] = CONFIG["expanded_profiles"][:2]

# Choices inherited from v4, visible and editable as a regular dictionary.
print("Plan choices:", json.dumps(CONFIG["choices"], indent=2))
validate_config(CONFIG)
OUTPUT = ROOT / "results" / RUN_NAME
print("Output:", OUTPUT)
print(json.dumps(CONFIG, indent=2))


## Train / evaluate / resume

This is the only cell that starts the experiment. Completed episodes save
weights, optimizer state and progress. A partial episode retries on resume;
completed candidate scores and evaluation traces are retained. Restart the kernel
after pulling code. Source/configuration changes require a new run directory.


In [ ]:
result = run(config=CONFIG, output=OUTPUT, max_minutes=MAX_MINUTES)
print(json.dumps(result, indent=2))


## Saved progress and training diagnostics (read only)


In [ ]:
print(json.dumps(status(OUTPUT), indent=2))
if (OUTPUT / "learning_summary.json").exists():
    learning = read_json(OUTPUT / "learning_summary.json")
    display(pd.DataFrame([{k: v for k, v in row.items() if k != "history"}
                          for row in learning["seeds"]]))
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
    for row in learning["seeds"]:
        history = row["history"]
        if history:
            ax.plot([x["episode"] for x in history], [x["mean_reward"] for x in history],
                    label="Seed " + str(row["seed"]))
    ax.set_title("Running training reward (not held-out performance)")
    ax.set_xlabel("Completed episodes")
    if ax.lines:
        ax.legend()
    plt.show()


## Held-out results

Show TEST separately from VALIDATION. Larger reward alone does not prove budget
or deadline satisfaction. Inspect censoring and screens, and retain every RL seed.
The supervised control uses the same observed training feedback as RL seed 11.


In [ ]:
SPLIT = "test"  # Or "validation"; never combine silently.
if (OUTPUT / "summary.csv").exists():
    results = pd.read_csv(OUTPUT / "summary.csv")
    results = results[results["split"] == SPLIT]
    display(results[["case_id", "method", "restricted_mean_s", "mean_cost_usd",
                     "censored_traces", "statistical_screen_passed", "planning_s",
                     "active_nodes", "shards", "replicas", "checkpoint_every",
                     "checkpoint_mode", "checkpoint_tier"]])
else:
    print("Training or evaluation is still in progress.")


## Read-only comparison plots

Only cases with every method evaluated enter these plots. Times and costs remain
restricted/observed if censored. Search timings include prerequisite heuristics;
learned inference includes candidate construction. Failure-unaware timing is
attributed healthy scoring only. Training cost is reported separately above.


In [ ]:
PROFILE = "reference"  # Also slow_io, slow_network, variable_repair.
REFERENCE_NODES = 8
if (OUTPUT / "summary.json").exists():
    table = plot_results(OUTPUT, profile=PROFILE, reference_nodes=REFERENCE_NODES, split=SPLIT)
    display(table)


## Compact analysis export

Share this ZIP after the experiment completes or pauses. It includes paired
comparisons, selected plans, policy hashes and learning history. Raw simulation
observations, candidate audits and model/optimizer checkpoints remain local.


In [ ]:
archive = export(OUTPUT)
print("Share:", archive)
print("Size (MiB):", round(archive.stat().st_size / 1024**2, 3))
